In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
os.chdir(r"C:\Users\anish\flight_disruption_project")

df       = pd.read_csv('data/raw/flights.csv', low_memory=False)
airlines = pd.read_csv('data/raw/airlines.csv')
airports = pd.read_csv('data/raw/airports.csv')

print("Loaded:", df.shape)

Loaded: (5819079, 31)


In [3]:
df['ARRIVAL_DELAY']   = pd.to_numeric(df['ARRIVAL_DELAY'], errors='coerce').fillna(0)
df['DEPARTURE_DELAY'] = pd.to_numeric(df['DEPARTURE_DELAY'], errors='coerce').fillna(0)
df['CANCELLED']       = pd.to_numeric(df['CANCELLED'], errors='coerce').fillna(0).astype(int)

# Fill delay reason columns
delay_cols = ['AIR_SYSTEM_DELAY', 'WEATHER_DELAY', 'AIRLINE_DELAY',
              'LATE_AIRCRAFT_DELAY', 'SECURITY_DELAY']
for col in delay_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print("Cleaning done")
print(df[['ARRIVAL_DELAY','DEPARTURE_DELAY','CANCELLED']].describe())

Cleaning done
       ARRIVAL_DELAY  DEPARTURE_DELAY     CANCELLED
count   5.819079e+06     5.819079e+06  5.819079e+06
mean    4.327482e+00     9.231431e+00  1.544643e-02
std     3.891956e+01     3.682282e+01  1.233201e-01
min    -8.700000e+01    -8.200000e+01  0.000000e+00
25%    -1.300000e+01    -5.000000e+00  0.000000e+00
50%    -5.000000e+00    -1.000000e+00  0.000000e+00
75%     7.000000e+00     7.000000e+00  0.000000e+00
max     1.971000e+03     1.988000e+03  1.000000e+00


In [4]:
flight_counts = df['DESTINATION_AIRPORT'].value_counts()
def assign_tier(airport):
    count = flight_counts.get(airport, 0)
    if count > 20000:   return 'large_hub'
    elif count > 5000:  return 'medium_hub'
    elif count > 1000:  return 'small_hub'
    else:               return 'non_hub'
df['airport_tier'] = df['DESTINATION_AIRPORT'].apply(assign_tier)
mct_map = {
    'large_hub':  90,
    'medium_hub': 75,
    'small_hub':  60,
    'non_hub':    45
}
np.random.seed(42)
df['connection_buffer'] = (
    df['airport_tier'].map(mct_map) +
    np.random.normal(0, 20, len(df))
).clip(15, 300).round(1)
df['missed_connection'] = (
    df['connection_buffer'] < df['ARRIVAL_DELAY']
).astype(int)

print("Connection buffer done")
print("Missed connection rate:", round(df['missed_connection'].mean() * 100, 2), "%")
print(df[['connection_buffer', 'missed_connection']].describe())

Connection buffer done
Missed connection rate: 3.88 %
       connection_buffer  missed_connection
count       5.819079e+06       5.819079e+06
mean        8.491820e+01       3.883261e-02
std         2.246936e+01       1.931959e-01
min         1.500000e+01       0.000000e+00
25%         7.040000e+01       0.000000e+00
50%         8.570000e+01       0.000000e+00
75%         1.002000e+02       0.000000e+00
max         1.907000e+02       1.000000e+00


In [5]:
df['SCHEDULED_DEPARTURE'] = pd.to_numeric(
    df['SCHEDULED_DEPARTURE'], errors='coerce'
).fillna(0)

df['scheduled_hour'] = (df['SCHEDULED_DEPARTURE'] // 100).astype(int)
def propagation_risk(hour):
    if hour < 10:   return 0   # Low risk
    elif hour < 15: return 1   # Medium risk
    else:           return 2   # High risk

df['propagation_risk'] = df['scheduled_hour'].apply(propagation_risk)

print("Propagation risk done")
print(df['propagation_risk'].value_counts())

Propagation risk done
propagation_risk
2    2366615
1    1778563
0    1673901
Name: count, dtype: int64


In [6]:
route_delay = df.groupby(
    ['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
)['ARRIVAL_DELAY'].mean()

df['route_congestion_index'] = df.set_index(
    ['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
).index.map(route_delay).values

df['route_congestion_index'] = df['route_congestion_index'].fillna(
    df['ARRIVAL_DELAY'].mean()
)

print("Route congestion index done")
print(df['route_congestion_index'].describe())

Route congestion index done
count    5.819079e+06
mean     4.327482e+00
std      4.879990e+00
min     -4.600000e+01
25%      1.491473e+00
50%      4.359060e+00
75%      7.454665e+00
max      3.810000e+02
Name: route_congestion_index, dtype: float64


In [7]:
def compute_disruption(row):
    score = 0
    if row['ARRIVAL_DELAY'] > 30:        score += 1
    if row['CANCELLED'] == 1:             score += 2
    if row['missed_connection'] == 1:     score += 2
    if row['ARRIVAL_DELAY'] > 120:        score += 1
    return 1 if score >= 2 else 0

df['DISRUPTION'] = df.apply(compute_disruption, axis=1)

counts = df['DISRUPTION'].value_counts()
ratios = df['DISRUPTION'].value_counts(normalize=True) * 100

print("Disruption counts:")
print(counts)
print("\nDisruption ratio:")
print(ratios.round(2))

Disruption counts:
DISRUPTION
0    5502508
1     316571
Name: count, dtype: int64

Disruption ratio:
DISRUPTION
0    94.56
1     5.44
Name: proportion, dtype: float64


In [8]:
def risk_tier(row):
    if row['ARRIVAL_DELAY'] <= 15 and row['CANCELLED'] == 0:
        return 'Low'
    elif row['ARRIVAL_DELAY'] <= 60 and row['CANCELLED'] == 0:
        return 'Medium'
    else:
        return 'High'

df['RISK_TIER'] = df.apply(risk_tier, axis=1)

print("Risk tier distribution:")
print(df['RISK_TIER'].value_counts())
print()
print(df['RISK_TIER'].value_counts(normalize=True).mul(100).round(2))

Risk tier distribution:
RISK_TIER
Low       4705697
Medium     704406
High       408976
Name: count, dtype: int64

RISK_TIER
Low       80.87
Medium    12.11
High       7.03
Name: proportion, dtype: float64


In [10]:
import os
os.makedirs('data/processed', exist_ok=True)
print("Folder created!")

features_to_keep = [
    'DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'CANCELLED',
    'DAY_OF_WEEK', 'DISTANCE',
    'AIR_SYSTEM_DELAY', 'WEATHER_DELAY',
    'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY',
    'connection_buffer', 'missed_connection',
    'propagation_risk', 'route_congestion_index',
    'DISRUPTION', 'RISK_TIER'
]

df_clean = df[features_to_keep].dropna()

print("Final dataset shape:", df_clean.shape)
print("Disruption balance:\n", df_clean['DISRUPTION'].value_counts())

# Save
df_clean.to_csv('data/processed/processed_flights.csv', index=False)
print("\nSaved to data/processed/processed_flights.csv")

Folder created!
Final dataset shape: (5819079, 15)
Disruption balance:
 DISRUPTION
0    5502508
1     316571
Name: count, dtype: int64

Saved to data/processed/processed_flights.csv
